In [27]:
import pandas as pd
import numpy as np
import requests
import datetime as dt

base_url ="https://fapi.binance.com/" 
url_klines = "fapi/v1/continuousKlines"
url_exchange_info = "/fapi/v1/exchangeInfo"

n = 65

info = {'marginAsset':'USDT',
        'contractType':'PERPETUAL',
         }

columns_df = ['open_time',
              'open',
              'high',
              'low',
              'close',
              'volume',
              'close_time',
              'quote_asset_volume',
              'number_of_trades',
              'taker_buy_volume',
              'taker_buy_quote_asset_volume',
              'ignore']
def get_data(client,symbol,n):
    params = {
        'pair': f'{symbol}',
        'contractType': 'PERPETUAL',
        'interval': '1d',  # Intervalo das velas (exemplo: 1h para uma hora)
        'limit': n         # Número de velas a serem retornadas
    }
    res = client.get(base_url + url_klines, params=params)
    return symbol,res.json()

def get_exchange_info(info):
    def filter(r,info=info):
        check = []
        for k,v in info.items():
            check.append(r.get(k) == v)
        if all(check):
            return True
        else:
            return False
        
    res = requests.get(base_url + url_exchange_info)
    tradable_symbols = res.json()['symbols']
    filtered_symbols = [r for r in tradable_symbols if filter(r,info)]

    return filtered_symbols

def get_data(symbol,n):
    params = {
        'pair': f'{symbol}',
        'contractType': 'PERPETUAL',
        'interval': '1d',  # Intervalo das velas (exemplo: 1h para uma hora)
        'limit': n         # Número de velas a serem retornadas
    }
    res = requests.get(base_url + url_klines, params=params)
    return symbol,res.json()

In [28]:
# todos os tickers de futuros
exchange_info = pd.DataFrame(get_exchange_info(info))

In [29]:
ele = {}
for c in exchange_info.columns:
    try:
        ele[c] = exchange_info[c].unique()
    except:
        print('herror',c)

herror underlyingSubType
herror filters
herror orderTypes
herror timeInForce


In [30]:
import matplotlib
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm

In [31]:
symbol_space = ele['symbol']

In [32]:
results = []
for s in symbol_space[:]:
    x = get_data(s,1200)
    results.append(x)
    
symbols_price_info = {symbol_info[0]:symbol_info[1] for symbol_info in results}

closing_prices = []
for s,prices in list(symbols_price_info.items()):
    df_price = pd.DataFrame(prices,columns=columns_df).dropna()
    df_price = df_price[['close_time','close']].copy()
    df_price.set_index('close_time',inplace=True)
    df_price.index = pd.to_datetime(df_price.index,unit='ms')
    closing_series = pd.to_numeric(df_price['close'])
    closing_series.name = s
    closing_prices.append(closing_series)

KeyboardInterrupt: 

In [ ]:
final_df = pd.DataFrame(closing_prices).T


In [19]:
final_df.to_csv("./assets/closing_prices.csv",sep=';')